# Module 4: Datasets and Experiments

> Self-directed module, ~30 min.

Continuing from Module 2: the agent runs and produces traces. This module covers the **offline** evaluation pattern — define a fixed dataset of examples and run on-demand experiments against it. Three evaluation patterns, ordered broadest to most granular:

1. **Final-response** — does the end-to-end answer satisfy the rubric? (LLM-as-judge)
2. **Single-step** — did the agent make the correct first decision? (does the first tool call match the expected one)
3. **Trajectory** — does the full sequence of tool calls match the expected trajectory? (exact match plus extra/missing counts)

All three run against the same dataset; only the run function and evaluators change. Online evaluations (Module 5) reuse the same judge pattern on every new trace as it lands, instead of running on a fixed dataset.

## Setup


In [ ]:
import sys
from pathlib import Path
project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import os
import time
from datetime import datetime, timedelta, timezone
from langsmith import Client, uuid7

from utils.config import load_active_agent, PROJECT_NAME

# Single source of truth for project name; ensure tracing is on.
os.environ["LANGSMITH_PROJECT"] = PROJECT_NAME
os.environ.setdefault("LANGSMITH_TRACING", "true")

client = Client()
agent = load_active_agent()

print(f"Project:  {PROJECT_NAME}")
print(f"Tracing:  {os.environ.get('LANGSMITH_TRACING')}")
print(f"Endpoint: {os.environ.get('LANGSMITH_ENDPOINT', 'https://api.smith.langchain.com')}")

from utils.models import model
from typing_extensions import TypedDict, Annotated
from langchain_core.messages import SystemMessage, HumanMessage
from collections import Counter
from typing import Any


## Part 1. Dataset

A LangSmith dataset is a versioned collection of examples, each with `inputs` and (optionally) `outputs`. The outputs are reference data for evaluators — typically a success rubric for LLM judges, or expected tool sequences for trajectory evaluation. The dataset is browsable and editable in the UI after creation; the SDK is for programmatic creation and scripting.

Five examples below cover the agent's main paths: profile-only, web-research-only, profile + research (× 2), and the unknown-client fall-through. Each example carries three reference fields so the same dataset drives all three evaluation patterns:

- `reference_answer` — a success rubric (not verbatim expected text) for the LLM judge
- `expected_first_tool` — the first tool call we expect the agent to make
- `trajectory` — the full expected sequence of tool calls


In [ ]:
examples = [
    {
        "inputs": {"query": "What does Smith Family Office hold?"},
        "outputs": {
            "reference_answer": "Lists Smith Family Office's top holdings (NVDA, GOOGL, BRK.B) with weights and the aggressive-growth risk profile.",
            "expected_first_tool": "get_client_profile",
            "trajectory": ["get_client_profile"],
        },
    },
    {
        "inputs": {"query": "Summarize recent news on Apple. Search at most once."},
        "outputs": {
            "reference_answer": "A brief, cited summary of recent Apple news from a web search.",
            "expected_first_tool": "task",
            "trajectory": ["task"],
        },
    },
    {
        "inputs": {"query": "Prep me for my meeting with Acme Pension Fund. Cover one major holding. Search at most once."},
        "outputs": {
            "reference_answer": "Acme Pension Fund's holdings and risk profile, plus recent news on one major holding (e.g., AAPL, MSFT, JPM, or XOM), with at least one cited source.",
            "expected_first_tool": "get_client_profile",
            "trajectory": ["get_client_profile", "task"],
        },
    },
    {
        "inputs": {"query": "Look up TechCorp Treasury, then research one of their holdings briefly. Search at most once."},
        "outputs": {
            "reference_answer": "TechCorp Treasury's conservative, fixed-income-heavy profile, plus recent context on one holding.",
            "expected_first_tool": "get_client_profile",
            "trajectory": ["get_client_profile", "task"],
        },
    },
    {
        "inputs": {"query": "Who is the client 'unknown-fund'?"},
        "outputs": {
            "reference_answer": "Tells the user the client is not in the system and lists the known client IDs.",
            "expected_first_tool": "get_client_profile",
            "trajectory": ["get_client_profile"],
        },
    },
]

dataset_name = "client-research-evals"

if client.has_dataset(dataset_name=dataset_name):
    existing = client.read_dataset(dataset_name=dataset_name)
    client.delete_dataset(dataset_id=existing.id)
    print(f"Deleted existing dataset '{dataset_name}'")

dataset = client.create_dataset(dataset_name)
client.create_examples(
    inputs=[e["inputs"] for e in examples],
    outputs=[e["outputs"] for e in examples],
    dataset_id=dataset.id,
)
print(f"Created dataset '{dataset_name}' with {len(examples)} examples")
print(f"View: {dataset.url}")


### Verify in LangSmith

Open `dataset.url` and confirm all five examples are present. The UI is where examples get edited, added, or split into train/test groups — once iteration starts, expect to mostly work with this dataset from the UI rather than re-running this cell.

<img src="../images/dataset.png" alt="LangSmith dataset detail view with five examples" style="width: auto; max-height: 360px; border-radius: 8px;">


## Part 2. Final-Response Evaluation

<img src="../images/final-response.png" style="width: auto; max-height: 280px; border-radius: 8px;">

End-to-end evaluation: run the agent on each example's input, capture the final response, score it against the rubric with an LLM judge. The judge prompt explicitly tells the model to grade against criteria, not literal text match — agents that paraphrase or restructure shouldn't get penalized for valid wording differences.


The run function does two things — invokes the agent, then concatenates any files the agent wrote into the response string. This matters because the agent often saves the actual content to `/client_brief.md` and replies with a one-liner like "Brief saved." Without the file dump, the judge would see only the one-liner and grade harshly.


In [ ]:
def run_agent_final(inputs: dict) -> dict:
    """Target function: run the agent and return its final response, with any saved files appended."""
    config = {"configurable": {"thread_id": str(uuid7())}}
    result = agent.invoke(
        {"messages": [{"role": "user", "content": inputs["query"]}]},
        config=config,
    )

    response = result["messages"][-1].content

    file_dump = []
    for path, file_data in (result.get("files") or {}).items():
        content = file_data
        if isinstance(file_data, dict) and "content" in file_data:
            content = file_data["content"]
        if isinstance(content, list):
            content = "\n".join(content)
        file_dump.append(f"--- {path} ---\n{content}")
    if file_dump:
        response += "\n\nFiles written:\n" + "\n\n".join(file_dump)

    return {"response": response}


The code below defines a correctness evaluator using the **LLM-as-judge** pattern — one common approach where an LLM scores the agent's output against a rubric. `TypedDict` with `Annotated` field metadata gives the structured-output binding both type safety and inline documentation that the judge sees as part of the schema. The judge prompt explicitly distinguishes "meets the rubric" from "matches the rubric verbatim" — important whenever the agent's output style differs from how the rubric is phrased.


In [ ]:
class CorrectnessGrade(TypedDict):
    """Score whether the agent's response satisfied the rubric."""
    score: Annotated[bool, ..., "True if the response meets the success rubric"]
    reasoning: Annotated[str, ..., "One short sentence of reasoning"]

correctness_judge_prompt = """You are grading a client research assistant.

You will see the user's request, the assistant's final response, and a RUBRIC describing success.

The rubric is success criteria, not the expected response text. Mark score=True if the response
demonstrates the criteria were met — even if worded differently from the rubric. Mark score=False
only if the response clearly missed the task, hallucinated facts (especially client identities),
or refused without good reason.

Give one short sentence of reasoning either way.
"""

judge = model.with_structured_output(CorrectnessGrade)

def correctness_evaluator(inputs, outputs, reference_outputs):
    grade = judge.invoke([
        SystemMessage(content=correctness_judge_prompt),
        HumanMessage(content=(
            f"User request: {inputs['query']}\n\n"
            f"Assistant response: {outputs['response']}\n\n"
            f"Success rubric: {reference_outputs['reference_answer']}"
        )),
    ])
    return {"key": "correctness", "score": int(grade["score"]), "comment": grade["reasoning"]}


Expect each example to take roughly 10–30 seconds depending on how many tool calls the agent makes. With `max_concurrency=2` and five examples, the full experiment runs in a couple of minutes. The returned `results` object exposes `experiment_name` and a URL once `client.evaluate` resolves.


In [ ]:
results = client.evaluate(
    run_agent_final,
    data=dataset_name,
    evaluators=[correctness_evaluator],
    experiment_prefix="final-response",
    max_concurrency=2,
)
print(f"Experiment: {results.experiment_name}")


Open `dataset.url` and switch to the **Experiments** tab. The experiment just created appears at the top; clicking it opens the per-example view with the agent's response, the judge's score, and the judge's reasoning column-by-column. Re-running this cell appends a new experiment with the same prefix, so iterations stay grouped.

<img src="../images/experiment_result.png" alt="LangSmith experiment result view with per-example scores and judge reasoning" style="width: auto; max-height: 360px; border-radius: 8px;">


## Part 3. Single-Step Evaluation

<img src="../images/single-step.png" style="width: auto; max-height: 280px; border-radius: 8px;">

Final-response evaluation answers "did the output meet the bar." Single-step evaluation answers "where did things go right or wrong inside the agent." For the client research agent, the most diagnostic single decision is the **first tool call**: when a named client is in the query, did the agent call `get_client_profile` first? When the query is open-ended research, did it delegate to the `research-agent` subagent via `task`?

Why bother with this when final-response already covers correctness? Final-response is too coarse to diagnose drift. If a regression lands and the final-response score drops, single-step tells you whether the agent is now calling the wrong tool first (planning regression) or calling the right tool and producing a bad answer (generation regression).


In [ ]:
def extract_tool_calls(messages: list[Any]) -> list[str]:
    """Extract tool call names from messages in order."""
    tool_names = []
    for msg in messages:
        if getattr(msg, "tool_calls", None):
            tool_names.extend(tc["name"] for tc in msg.tool_calls)
    return tool_names

def run_agent_first_tool(inputs: dict) -> dict:
    """Single-step target: capture only the first tool the agent called."""
    config = {"configurable": {"thread_id": str(uuid7())}}
    result = agent.invoke(
        {"messages": [{"role": "user", "content": inputs["query"]}]},
        config=config,
    )
    tool_calls = extract_tool_calls(result["messages"])
    return {"first_tool": tool_calls[0] if tool_calls else None}

def first_tool_match(outputs, reference_outputs):
    """Exact match between the agent's first tool call and the expected one."""
    return {
        "key": "first_tool_match",
        "score": int(outputs["first_tool"] == reference_outputs["expected_first_tool"]),
    }


In [ ]:
results = client.evaluate(
    run_agent_first_tool,
    data=dataset_name,
    evaluators=[first_tool_match],
    experiment_prefix="single-step",
    max_concurrency=2,
)
print(f"Experiment: {results.experiment_name}")


In the Experiments tab, the `single-step` experiment shows up alongside the `final-response` one. The per-example column shows the agent's first tool vs the expected one — a `0` is a single-step regression even when final-response might still pass (the agent recovered downstream).


## Part 4. Trajectory Evaluation

<img src="../images/trajectory.png" style="width: auto; max-height: 280px; border-radius: 8px;">

Trajectory evaluation scores the full sequence of tool calls. Three metrics:

- **Exact match** — does the trajectory equal the reference, in order?
- **Extra steps** — how many tool calls did the agent make that weren't in the reference?
- **Missing steps** — how many expected tool calls did the agent skip?

Extra and missing surface drift even when exact match is too strict — useful when an agent has several valid paths to the same answer. Trajectory complements single-step: single-step catches the first divergence; trajectory catches *all* of them.


In [ ]:
def run_agent_trajectory(inputs: dict) -> dict:
    config = {"configurable": {"thread_id": str(uuid7())}}
    result = agent.invoke(
        {"messages": [{"role": "user", "content": inputs["query"]}]},
        config=config,
    )
    return {"trajectory": extract_tool_calls(result["messages"])}

def trajectory_match(outputs, reference_outputs):
    return {
        "key": "exact_match",
        "score": int(outputs["trajectory"] == reference_outputs["trajectory"]),
    }

def extra_steps(outputs, reference_outputs):
    extras = Counter(outputs["trajectory"]) - Counter(reference_outputs["trajectory"])
    return {"key": "extra_steps", "score": sum(extras.values())}

def missing_steps(outputs, reference_outputs):
    missing = Counter(reference_outputs["trajectory"]) - Counter(outputs["trajectory"])
    return {"key": "missing_steps", "score": sum(missing.values())}


In [ ]:
results = client.evaluate(
    run_agent_trajectory,
    data=dataset_name,
    evaluators=[trajectory_match, extra_steps, missing_steps],
    experiment_prefix="trajectory",
    max_concurrency=2,
)
print(f"Experiment: {results.experiment_name}")


### Reviewing experiments

The dataset's **Experiments** tab is the central place to inspect, compare, and iterate. All three experiments above — `final-response`, `single-step`, `trajectory` — appear there, each as a row with summary metrics (correctness rate, first-tool match rate, exact match rate, extras/missing) visible at a glance.

<img src="../images/experiments_page.png" alt="LangSmith Experiments tab listing multiple experiment runs against the dataset" style="width: auto; max-height: 360px; border-radius: 8px;">


### Compare experiments

On the dataset's **Experiments** tab, select two or more experiments and use **Compare** to put them side by side. The per-example diff highlights where one experiment scored differently from another — the fastest way to see whether a change to the agent moved the needle, or just moved a few examples around.

<img src="../images/experiment_comparison.png" alt="Experiment comparison view" style="width: auto; max-height: 360px; border-radius: 8px;">


### Set an experiment as baseline

Hovering over an experiment row reveals a **Set as baseline** action. Once set, the baseline pins to the top of the Experiments tab and every other experiment shows performance deltas against it across each metric column — a fast way to see whether the most recent run is an improvement, a regression, or a wash.

Useful whenever you're holding most variables constant and varying one thing (a prompt tweak, a model swap, a tool change) to measure its effect in isolation.

<img src="../images/set_baseline_button.png" alt="Setting an experiment as baseline via hover menu in LangSmith" style="display: block; margin: 0; width: auto; max-height: 360px; border-radius: 8px;">

<img src="../images/set_baseline.png" alt="Experiments list with baseline pinned and performance deltas surfaced per metric column" style="display: block; margin: 0; width: auto; max-height: 360px; border-radius: 8px;">


## Recap

| Pattern | What it scores | Reference field used |
|---|---|---|
| **Final-response** | End-to-end response against a success rubric (LLM-as-judge) | `reference_answer` |
| **Single-step** | First tool call decision | `expected_first_tool` |
| **Trajectory** | Full tool-call sequence (exact + extra + missing) | `trajectory` |

All three reuse the same dataset; only the run function and evaluator change. Run all three on every iteration of the agent — they catch different classes of regression and together give a complete diff between two versions.

Next: `05_online_evals.ipynb` — same judge, but applied automatically to every new trace.